In [ ]:

# ==========================
# Step 1  导入需要的工具
# ==========================

import pandas as pd
import numpy as np
import os

from google.colab import drive

In [ ]:
# ==========================
# Step 2  挂载Google Drive
# ==========================

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ==========================
# Step 3 查看Project文件夹
# ==========================

project = "/content/drive/MyDrive/RCEP_Trade_Network_Project/"

os.listdir(project)

['01_Data_Raw', '02_Data_Clean']

In [ ]:
# ==========================================
# Step 4  设置 Raw / Clean 文件夹
# ==========================================

RAW = project + "01_Data_Raw/"
CLEAN = project + "02_Data_Clean/"

print("RAW =", RAW)
print("CLEAN =", CLEAN)

RAW = /content/drive/MyDrive/RCEP_Trade_Network_Project/01_Data_Raw/
CLEAN = /content/drive/MyDrive/RCEP_Trade_Network_Project/02_Data_Clean/


In [ ]:
# ==========================================
# Step 5  读取国家代码
# ==========================================

country = pd.read_csv(
    RAW + "country_codes_V202601.csv"
)

country.head()

,country_code,country_name,country_iso2,country_iso3
0,4,Afghanistan,AF,AFG
1,8,Albania,AL,ALB
2,12,Algeria,DZ,DZA
3,16,American Samoa,AS,ASM
4,20,Andorra,AD,AND


In [ ]:
# ==========================================
# Step 6  定义RCEP成员国
# ==========================================

rcep_codes = [
    36,   # Australia
    96,   # Brunei Darussalam
    104,  # Myanmar
    116,  # Cambodia
    156,  # China
    360,  # Indonesia
    392,  # Japan
    410,  # Rep. of Korea
    418,  # Laos
    458,  # Malaysia
    554,  # New Zealand
    608,  # Philippines
    702,  # Singapore
    704,  # Viet Nam
    764   # Thailand
]

country[
    country["country_code"].isin(rcep_codes)
]

,country_code,country_name,country_iso2,country_iso3
9,36,Australia,AU,AUS
28,96,Brunei Darussalam,BN,BRN
30,104,Myanmar,MM,MMR
33,116,Cambodia,KH,KHM
42,156,China,CN,CHN
96,360,Indonesia,ID,IDN
104,392,Japan,JP,JPN
109,410,Rep. of Korea,KR,KOR
112,418,Lao People's Dem. Rep.,LA,LAO
123,458,Malaysia,MY,MYS


In [ ]:
# ==========================================
# Step 7  批量生成2022-2024贸易网络
# ==========================================

years = [2022, 2023, 2024]

for year in years:

    print("=" * 40)
    print(f"Processing {year}...")
    print("=" * 40)

    # 读取BACI数据
    df = pd.read_csv(
        RAW + f"BACI_HS22_Y{year}_V202601.csv"
    )

    # 保留RCEP成员之间贸易
    rcep_trade = df[
        (df["i"].isin(rcep_codes)) &
        (df["j"].isin(rcep_codes))
    ]

    # 汇总国家之间贸易额
    edge = (
        rcep_trade
        .groupby(["i", "j"], as_index=False)["v"]
        .sum()
    )

    # Gephi格式
    edge = edge.rename(columns={
        "i": "Source",
        "j": "Target",
        "v": "Weight"
    })

    # 保存
    edge.to_csv(
        CLEAN + f"RCEP_EdgeList_{year}.csv",
        index=False
    )

    print(f"原始数据：{len(df):,} 行")
    print(f"RCEP数据：{len(rcep_trade):,} 行")
    print(f"Edge数量：{len(edge)}")
    print("保存成功！\n")

Processing 2022...
原始数据：11,007,862 行
RCEP数据：386,231 行
Edge数量：208
保存成功！

Processing 2023...
原始数据：11,641,922 行
RCEP数据：403,574 行
Edge数量：210
保存成功！

Processing 2024...
原始数据：11,250,411 行
RCEP数据：395,171 行
Edge数量：207
保存成功！



In [ ]:
# ==========================================
# Step 8  查看输出文件
# ==========================================

os.listdir(CLEAN)

['RCEP_EdgeList_2024.csv', 'RCEP_EdgeList_2022.csv', 'RCEP_EdgeList_2023.csv']